In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import json
from typing import Any, Dict, List, Tuple, Union, Optional
import re
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier, LGBMRegressor

import lightgbm as lgb
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)

# Carregamento da base

In [2]:
caminho = Path('../outputs/base_gm_tratada.parquet')
dados = pd.read_parquet(caminho)

# Configuração pré-execução do modelo

In [3]:
TARGET_COL = "pontos_num"

CATEGORICAL_COLS = [
    "atleta_id",
    "clube_id",
    "adversario_id",
    "posicao_id",
    "rodada_id",
    "ano",
]

DROP_COLS = [
    TARGET_COL,
    'A',
    'CA',
    'CV',
    'DE',
    'DP',
    'DS',
    'FC',
    'FD',
    'FF',
    'FS',
    'FT',
    'G',
    'GC',
    'GS',
    'I',
    'PC',
    'PP',
    'PS',
    'SG',
    'entrou_em_campo',
    'equipe_visitante_id',
]

COL_TEMPORADA = "ano"
COL_RODADA = "rodada_id"

TEMPORADA_RODADA_CORTE_TREINO = (2024, 38)   # último round incluído no treino
TEMPORADA_RODADA_CORTE_VALID = (2025, 19)    # último round incluído na validação

# Split temporal - Treino, validação e teste

In [4]:
chave = pd.Series(
    list(zip(dados[COL_TEMPORADA], dados[COL_RODADA])),
    index=dados.index,
)

treino_mask = chave <= TEMPORADA_RODADA_CORTE_TREINO
valid_mask = (chave > TEMPORADA_RODADA_CORTE_TREINO) & (chave <= TEMPORADA_RODADA_CORTE_VALID)
teste_mask = chave > TEMPORADA_RODADA_CORTE_VALID

df_treino = dados.loc[treino_mask].copy()
df_valid = dados.loc[valid_mask].copy()
df_teste = dados.loc[teste_mask].copy()

print(f"Treino:    {df_treino.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_TREINO}")
print(f"Validação: {df_valid.shape[0]:>7} linhas | até {TEMPORADA_RODADA_CORTE_VALID}")
print(f"Teste:     {df_teste.shape[0]:>7} linhas | após {TEMPORADA_RODADA_CORTE_VALID}")

Treino:      82927 linhas | até (2024, 38)
Validação:   13552 linhas | até (2025, 19)
Teste:       13473 linhas | após (2025, 19)


# Slip das features e target - Treino, validação e teste

In [5]:
y_treino = df_treino[TARGET_COL].copy()
X_treino = df_treino.drop(columns=[c for c in DROP_COLS if c in df_treino.columns]).copy()

y_valid = df_valid[TARGET_COL].copy()
X_valid = df_valid.drop(columns=[c for c in DROP_COLS if c in df_valid.columns]).copy()

y_teste = df_teste[TARGET_COL].copy()
X_teste = df_teste.drop(columns=[c for c in DROP_COLS if c in df_teste.columns]).copy()

for col in CATEGORICAL_COLS:
    if col in X_treino.columns:
        X_treino[col] = X_treino[col].astype("category")
    if col in X_valid.columns:
        X_valid[col] = X_valid[col].astype("category")
    if col in X_teste.columns:
        X_teste[col] = X_teste[col].astype("category")

# Treinamento com early-stopping

In [6]:
melhores_params = {
    'subsample': 0.6,
    'reg_lambda': 0.5,
    'reg_alpha': 0.5,
    'num_leaves': 15.0,
    'min_child_samples': 30.0,
    'max_depth': 4.0,
    'learning_rate': 0.05,
    'colsample_bytree': 0.6
 }

melhores_params["num_leaves"] = int(melhores_params["num_leaves"])
melhores_params["max_depth"] = int(melhores_params["max_depth"])
melhores_params["min_child_samples"] = int(melhores_params["min_child_samples"])
params_finais = dict(
    objective="regression",
    metric="rmse",
    n_estimators=5000,
    random_state=42,
    n_jobs=-1,
    **melhores_params,
)

modelo = lgb.LGBMRegressor(**params_finais)

cat_features = [c for c in CATEGORICAL_COLS if c in X_treino.columns]

modelo.fit(
    X_treino, y_treino,
    eval_set=[(X_treino, y_treino), (X_valid, y_valid)],
    eval_names=["treino", "valid"],
    eval_metric="rmse",
    categorical_feature=cat_features,
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


/Users/pedro.goncalves/Desktop/cartola-prediction-points/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 26388
[LightGBM] [Info] Number of data points in the train set: 82927, number of used features: 207
[LightGBM] [Info] Start training from score 1.323612
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

,num_leaves,15
,max_depth,4
,learning_rate,0.05
,n_estimators,5000
,objective,'regression'
,min_child_samples,30
,subsample,0.6
,colsample_bytree,0.6
,reg_alpha,0.5
,reg_lambda,0.5
,random_state,42


# Predição

In [7]:
y_pred_valid = modelo.predict(X_valid, num_iteration=modelo.best_iteration_)
data_predicao_valid = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

y_pred_teste = modelo.predict(X_teste, num_iteration=modelo.best_iteration_)
data_predicao_teste = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Dataframes com real vs. predito, preservando o índice original (útil para
# rastrear de volta ao atleta_id / rodada_id / ano na base original)
df_pred_valid = df_valid[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_valid["y_real"] = y_valid.values
df_pred_valid["y_predito"] = y_pred_valid
df_pred_valid["erro"] = df_pred_valid["y_real"] - df_pred_valid["y_predito"]
df_pred_valid["data_predicao"] = data_predicao_valid

df_pred_teste = df_teste[[COL_TEMPORADA, COL_RODADA, "atleta_id", "posicao_id", "clube_id"]].copy()
df_pred_teste["y_real"] = y_teste.values
df_pred_teste["y_predito"] = y_pred_teste
df_pred_teste["erro"] = df_pred_teste["y_real"] - df_pred_teste["y_predito"]
df_pred_teste["data_predicao"] = data_predicao_teste

print("\n--- Amostra de predições (validação) ---")
print(df_pred_valid.head(10).to_string(index=False))

print("\n--- Amostra de predições (teste) ---")
print(df_pred_teste.head(10).to_string(index=False))


--- Amostra de predições (validação) ---
 ano  rodada_id  atleta_id  posicao_id  clube_id  y_real  y_predito      erro        data_predicao
2025          1      10066           4       114    1.20   2.288089 -1.088089 2026-09-10T17:13:56Z
2025          1      10082           2       101    8.50   3.835414  4.664586 2026-09-10T17:13:56Z
2025          1      10104           4       114    1.00   2.793027 -1.793027 2026-09-10T17:13:56Z
2025          1      10156           3       101    0.00   0.556341 -0.556341 2026-09-10T17:13:56Z
2025          1      10174           5       114   -0.60   0.465558 -1.065558 2026-09-10T17:13:56Z
2025          1      10178           4       114    9.80   2.558550  7.241450 2026-09-10T17:13:56Z
2025          1      10230           3       114    0.00   0.576351 -0.576351 2026-09-10T17:13:56Z
2025          1      10296           6       114    4.33   3.204085  1.125915 2026-09-10T17:13:56Z
2025          1      10346           5       101    5.10   4.970495

# Avaliação e métricas de desempenho

In [8]:
rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
r2_valid = r2_score(y_valid, y_pred_valid)

try:
    mape_valid = mean_absolute_percentage_error(y_valid, y_pred_valid)
except Exception:
    mape_valid = np.nan

metricas_valid = {"RMSE": rmse_valid, "MAE": mae_valid, "R2": r2_valid}

print("\n--- Métricas (validação) ---")
for nome, valor in metricas_valid.items():
    print(f"{nome:>6}: {valor:.4f}")

rmse_teste = np.sqrt(mean_squared_error(y_teste, y_pred_teste))
mae_teste = mean_absolute_error(y_teste, y_pred_teste)
r2_teste = r2_score(y_teste, y_pred_teste)
try:
    mape_teste = mean_absolute_percentage_error(y_teste, y_pred_teste)
except Exception:
    mape_teste = np.nan

metricas_teste = {"RMSE": rmse_teste, "MAE": mae_teste, "R2": r2_teste}

print("\n--- Métricas (teste) ---")
for nome, valor in metricas_teste.items():
    print(f"{nome:>6}: {valor:.4f}")


--- Métricas (validação) ---
  RMSE: 2.5605
   MAE: 1.4581
    R2: 0.3292

--- Métricas (teste) ---
  RMSE: 2.5595
   MAE: 1.3970
    R2: 0.3139


O dicionário "melhores_params" com os hiper-parâmetros otimizados foi gerado pelo bloco de cross-validation ao final desse notebook.

# Exportação das predições

In [10]:
registros = []
for row in df_pred_teste.itertuples(index=False):
    registro = {
        "atleta_id": row.atleta_id,
        "ano": row.ano,
        "rodada_id": row.rodada_id,
        "clube_id": row.clube_id,
        "posicao_id": row.posicao_id,
        "pontos_predito": round(float(row.y_predito), 2),
        "data_predicao": row.data_predicao,
    }
    registros.append(registro)

payload = {"previsoes": registros}

CAMINHO_SAIDA = Path(f"../outputs/previsoes_{datetime.now(timezone.utc).strftime("%Y-%m-%d-%H:%M:%SZ")}.json")

with open(CAMINHO_SAIDA, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"\n{len(registros)} predições exportadas para '{CAMINHO_SAIDA}'.")

# Amostra do JSON gerado (primeiros 2 registros), só para conferência visual
print("\n--- Amostra do JSON exportado ---")
print(json.dumps({"previsoes": registros[:2]}, ensure_ascii=False, indent=2))


13473 predições exportadas para '../outputs/previsoes_2026-09-10-17:13:56Z.json'.

--- Amostra do JSON exportado ---
{
  "previsoes": [
    {
      "atleta_id": 10008,
      "ano": 2025,
      "rodada_id": 20,
      "clube_id": 105,
      "posicao_id": 1,
      "pontos_predito": 3.91,
      "data_predicao": "2026-09-10T17:13:56Z"
    },
    {
      "atleta_id": 10009,
      "ano": 2025,
      "rodada_id": 20,
      "clube_id": 105,
      "posicao_id": 3,
      "pontos_predito": 0.48,
      "data_predicao": "2026-09-10T17:13:56Z"
    }
  ]
}


# FOLDS WALK-FORWARD - Cross-validation

Esse bloco se encontra comentado devido ao tempo que leva para ser executado por completo.

In [11]:
# # ---------------------------------------------------------------------------
# # 1. BASE DE OTIMIZAÇÃO (treino + validação, teste fica de fora)
# # ---------------------------------------------------------------------------

# df_cv = pd.concat([df_treino, df_valid], ignore_index=False).sort_values(
#     [COL_TEMPORADA, COL_RODADA]
# )

# # Mapeia cada combinação única (ano, rodada_id) para um índice sequencial
# # de "período" -> isso vira a unidade de corte temporal (não a linha individual)
# periodos = (
#     df_cv[[COL_TEMPORADA, COL_RODADA]]
#     .drop_duplicates()
#     .sort_values([COL_TEMPORADA, COL_RODADA])
#     .reset_index(drop=True)
# )
# periodos["periodo_id"] = periodos.index

# df_cv = df_cv.merge(periodos, on=[COL_TEMPORADA, COL_RODADA], how="left")

# total_periodos = periodos.shape[0]
# print(f"Total de períodos (ano, rodada) disponíveis para CV: {total_periodos}")


# # ---------------------------------------------------------------------------
# # 2. GERAÇÃO DOS FOLDS WALK-FORWARD (expanding window, por blocos de rodada)
# # ---------------------------------------------------------------------------

# N_SPLITS = 5              # número de folds de validação
# N_RODADAS_TESTE_FOLD = 5  # tamanho da janela de teste de cada fold, em rodadas
# MIN_RODADAS_TREINO = 60   # tamanho mínimo do treino no primeiro fold

# folds = []
# inicio_teste = MIN_RODADAS_TREINO

# for _ in range(N_SPLITS):
#     fim_teste = inicio_teste + N_RODADAS_TESTE_FOLD
#     if fim_teste > total_periodos:
#         break
#     train_periodo_ids = list(range(0, inicio_teste))
#     test_periodo_ids = list(range(inicio_teste, fim_teste))
#     folds.append((train_periodo_ids, test_periodo_ids))
#     inicio_teste = fim_teste  # expanding window: próximo treino inclui esse bloco de teste

# print(f"{len(folds)} fold(s) de walk-forward gerados (de {N_SPLITS} solicitados).")
# for i, (tr, te) in enumerate(folds):
#     print(
#         f"  Fold {i}: treino = períodos [0, {tr[-1]}] ({len(tr)} rodadas) "
#         f"| teste = períodos [{te[0]}, {te[-1]}] ({len(te)} rodadas)"
#     )

# if len(folds) == 0:
#     raise ValueError(
#         "Nenhum fold foi gerado. Reduza MIN_RODADAS_TREINO, N_RODADAS_TESTE_FOLD "
#         "ou N_SPLITS - não há períodos suficientes na base de CV."
#     )


# # ---------------------------------------------------------------------------
# # 3. ESPAÇO DE BUSCA DE HIPERPARÂMETROS
# # ---------------------------------------------------------------------------

# param_grid = {
#     "num_leaves": [15, 31, 63, 127],
#     "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
#     "max_depth": [-1, 4, 6, 8, 10],
#     "min_child_samples": [10, 20, 30, 50, 100],
#     "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
#     "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
#     "reg_alpha": [0.0, 0.01, 0.1, 0.5, 1.0],
#     "reg_lambda": [0.0, 0.01, 0.1, 0.5, 1.0],
# }

# N_ITER = 30  # nº de combinações aleatórias testadas (ajuste conforme seu orçamento de tempo)

# sampler = ParameterSampler(param_grid, n_iter=N_ITER, random_state=42)


# # ---------------------------------------------------------------------------
# # 4. LOOP DE BUSCA: para cada combinação, treina e avalia em todos os folds
# # ---------------------------------------------------------------------------

# resultados = []

# for idx_comb, params_candidato in enumerate(sampler):
#     rmses_fold = []

#     for idx_fold, (train_ids, test_ids) in enumerate(folds):
#         df_fold_treino = df_cv[df_cv["periodo_id"].isin(train_ids)]
#         df_fold_teste = df_cv[df_cv["periodo_id"].isin(test_ids)]

#         y_fold_treino = df_fold_treino[TARGET_COL]
#         X_fold_treino = df_fold_treino.drop(
#             columns=[c for c in DROP_COLS if c in df_fold_treino.columns] + ["periodo_id"]
#         )

#         y_fold_teste = df_fold_teste[TARGET_COL]
#         X_fold_teste = df_fold_teste.drop(
#             columns=[c for c in DROP_COLS if c in df_fold_teste.columns] + ["periodo_id"]
#         )

#         for col in CATEGORICAL_COLS:
#             if col in X_fold_treino.columns:
#                 X_fold_treino[col] = X_fold_treino[col].astype("category")
#             if col in X_fold_teste.columns:
#                 X_fold_teste[col] = X_fold_teste[col].astype("category")

#         params_fold = dict(
#             objective="regression",
#             metric="rmse",
#             n_estimators=2000,  # teto alto; early stopping decide o real
#             random_state=42,
#             n_jobs=-1,
#             **params_candidato,
#         )

#         modelo_fold = lgb.LGBMRegressor(**params_fold)
#         cat_features_fold = [c for c in CATEGORICAL_COLS if c in X_fold_treino.columns]

#         modelo_fold.fit(
#             X_fold_treino, y_fold_treino,
#             eval_set=[(X_fold_teste, y_fold_teste)],
#             eval_metric="rmse",
#             categorical_feature=cat_features_fold,
#             callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
#         )

#         y_pred_fold = modelo_fold.predict(X_fold_teste, num_iteration=modelo_fold.best_iteration_)
#         rmse_fold = np.sqrt(mean_squared_error(y_fold_teste, y_pred_fold))
#         rmses_fold.append(rmse_fold)

#     rmse_medio = float(np.mean(rmses_fold))
#     rmse_std = float(np.std(rmses_fold))

#     resultados.append({**params_candidato, "rmse_medio": rmse_medio, "rmse_std": rmse_std})

#     print(
#         f"[{idx_comb + 1}/{N_ITER}] RMSE médio: {rmse_medio:.4f} (+/- {rmse_std:.4f}) "
#         f"| params: {params_candidato}"
#     )


# # ---------------------------------------------------------------------------
# # 5. RESULTADO FINAL: ranking das combinações e melhores hiperparâmetros
# # ---------------------------------------------------------------------------

# df_resultados = pd.DataFrame(resultados).sort_values("rmse_medio").reset_index(drop=True)

# print("\n--- Top 5 combinações (menor RMSE médio primeiro) ---")
# print(df_resultados.head(5).to_string(index=False))

# melhores_params = (
#     df_resultados.iloc[0]
#     .drop(labels=["rmse_medio", "rmse_std"])
#     .to_dict()
# )

In [12]:
# print("\nMelhores hiperparâmetros encontrados:")
# melhores_params